# Day 4: Policy Gradient Methods

**Reinforcement Learning: From Theory to Practice**

---

## Overview

Instead of learning a value function and deriving a policy, **policy gradient** methods parameterize the policy directly and optimize it via gradient ascent:

1. **REINFORCE** algorithm from scratch
2. **Baseline subtraction** for variance reduction
3. **Advantage Actor-Critic (A2C)** overview and implementation
4. **PPO with Stable-Baselines3**
5. **Algorithm comparison**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

import gymnasium as gym

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

## 1. The Policy Gradient Theorem

We parameterize the policy as $\pi_\theta(a|s)$ and maximize the expected return:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^T \gamma^t r_t\right]$$

The **policy gradient theorem** gives us:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

where $G_t = \sum_{k=t}^T \gamma^{k-t} r_k$ is the return from time $t$.

## 2. Policy Network

In [ ]:
class PolicyNetwork(nn.Module):
    """
    Simple policy network that outputs action probabilities.
    
    pi_theta(a | s) = softmax(f_theta(s))
    """
    
    def __init__(self, state_dim, n_actions, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions)
        )
    
    def forward(self, x):
        logits = self.net(x)
        return F.softmax(logits, dim=-1)
    
    def get_action(self, state):
        """Sample an action from the policy and return action + log probability."""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs = self.forward(state_t)
        dist = Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)


# Test
policy = PolicyNetwork(4, 2).to(device)
test_state = np.array([0.0, 0.1, 0.02, -0.03])
action, log_prob = policy.get_action(test_state)
print(f'Action: {action}, Log prob: {log_prob.item():.4f}')

## 3. REINFORCE (Vanilla Policy Gradient)

The simplest policy gradient algorithm:

1. Collect a full episode using $\pi_\theta$
2. For each step $t$, compute the return $G_t$
3. Update: $\theta \leftarrow \theta + \alpha \sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) G_t$

In [ ]:
def reinforce(env_name='CartPole-v1', n_episodes=500, gamma=0.99, lr=1e-3,
              use_baseline=False, seed=42):
    """
    REINFORCE algorithm with optional baseline subtraction.
    
    Parameters
    ----------
    use_baseline : bool
        If True, subtract a running average of returns as baseline.
    
    Returns
    -------
    episode_rewards : list of total rewards per episode
    """
    env = gym.make(env_name)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    state_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n
    
    policy = PolicyNetwork(state_dim, n_actions).to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    
    episode_rewards = []
    baseline = 0.0  # running average of returns
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        
        log_probs = []
        rewards = []
        
        done = False
        while not done:
            action, log_prob = policy.get_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            log_probs.append(log_prob)
            rewards.append(reward)
            state = next_state
        
        # Compute returns G_t for each step
        T = len(rewards)
        returns = np.zeros(T)
        G = 0.0
        for t in reversed(range(T)):
            G = rewards[t] + gamma * G
            returns[t] = G
        
        total_reward = sum(rewards)
        episode_rewards.append(total_reward)
        
        # Update baseline (exponential moving average)
        baseline = 0.99 * baseline + 0.01 * returns[0]
        
        # Normalize returns (optional but helps)
        returns_t = torch.FloatTensor(returns).to(device)
        if use_baseline:
            returns_t = returns_t - baseline
        
        # Normalize for stable gradients
        if returns_t.std() > 0:
            returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)
        
        # Compute policy gradient loss
        # Loss = -E[log pi(a|s) * G_t]  (negative because we want to maximize)
        policy_loss = []
        for log_prob, G_t in zip(log_probs, returns_t):
            policy_loss.append(-log_prob * G_t)
        
        loss = torch.stack(policy_loss).sum()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (ep + 1) % 100 == 0:
            avg = np.mean(episode_rewards[-100:])
            print(f'Episode {ep+1:4d} | Avg Reward (100): {avg:6.1f}')
    
    env.close()
    return episode_rewards


print('=== REINFORCE without baseline ===')
rewards_reinforce = reinforce(use_baseline=False, n_episodes=500)

In [ ]:
print('\n=== REINFORCE with baseline ===')
rewards_reinforce_bl = reinforce(use_baseline=True, n_episodes=500)

In [ ]:
# Compare with and without baseline
window = 20

def smooth(x, w):
    return np.convolve(x, np.ones(w)/w, mode='valid')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(smooth(rewards_reinforce, window), linewidth=2,
        label='REINFORCE', color='#2196F3')
ax.plot(smooth(rewards_reinforce_bl, window), linewidth=2,
        label='REINFORCE + Baseline', color='#FF5722')
ax.axhline(y=475, color='green', linestyle='--', alpha=0.5, label='Solved')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('REINFORCE: Effect of Baseline Subtraction')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Advantage Actor-Critic (A2C)

REINFORCE has high variance because it uses full episodic returns. **Actor-Critic** methods combine:
- **Actor**: policy network $\pi_\theta(a|s)$
- **Critic**: value network $V_\phi(s)$

The **advantage** is:
$$A(s_t, a_t) = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$$

This replaces the full return $G_t$ in the policy gradient, greatly reducing variance.

In [ ]:
class ActorCritic(nn.Module):
    """
    Actor-Critic network with shared feature layers.
    
    - Actor head: outputs action probabilities
    - Critic head: outputs state value V(s)
    """
    
    def __init__(self, state_dim, n_actions, hidden_dim=128):
        super(ActorCritic, self).__init__()
        
        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Actor head
        self.actor = nn.Linear(hidden_dim, n_actions)
        
        # Critic head
        self.critic = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        features = self.shared(x)
        action_probs = F.softmax(self.actor(features), dim=-1)
        value = self.critic(features)
        return action_probs, value
    
    def get_action(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs, value = self.forward(state_t)
        dist = Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action), value


def train_a2c(env_name='CartPole-v1', n_episodes=500, gamma=0.99, lr=3e-4,
              entropy_coeff=0.01, seed=42):
    """
    Advantage Actor-Critic (A2C) -- single-environment version.
    
    Returns
    -------
    episode_rewards : list of total rewards per episode
    """
    env = gym.make(env_name)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    state_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n
    
    model = ActorCritic(state_dim, n_actions).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    episode_rewards = []
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        
        log_probs = []
        values = []
        rewards = []
        entropies = []
        
        done = False
        while not done:
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            probs, value = model(state_t)
            dist = Categorical(probs)
            action = dist.sample()
            
            next_state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            
            log_probs.append(dist.log_prob(action))
            values.append(value)
            rewards.append(reward)
            entropies.append(dist.entropy())
            
            state = next_state
        
        episode_rewards.append(sum(rewards))
        
        # Compute returns and advantages
        T = len(rewards)
        returns = torch.zeros(T).to(device)
        G = 0.0
        for t in reversed(range(T)):
            G = rewards[t] + gamma * G
            returns[t] = G
        
        values_t = torch.cat(values).squeeze()
        log_probs_t = torch.stack(log_probs)
        entropies_t = torch.stack(entropies)
        
        # Advantage = returns - V(s)
        advantages = returns - values_t.detach()
        
        # Normalize advantages
        if advantages.std() > 0:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # Actor loss: -log pi(a|s) * A(s,a)
        actor_loss = -(log_probs_t * advantages).mean()
        
        # Critic loss: MSE between V(s) and returns
        critic_loss = F.mse_loss(values_t, returns)
        
        # Entropy bonus for exploration
        entropy_loss = -entropies_t.mean()
        
        # Total loss
        loss = actor_loss + 0.5 * critic_loss + entropy_coeff * entropy_loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        
        if (ep + 1) % 100 == 0:
            avg = np.mean(episode_rewards[-100:])
            print(f'Episode {ep+1:4d} | Avg Reward (100): {avg:6.1f}')
    
    env.close()
    return episode_rewards


print('=== Training A2C ===')
rewards_a2c = train_a2c(n_episodes=500)

In [ ]:
# Compare REINFORCE vs A2C
fig, ax = plt.subplots(figsize=(10, 5))

for rewards, label, color in [
    (rewards_reinforce_bl, 'REINFORCE + Baseline', '#2196F3'),
    (rewards_a2c, 'A2C', '#4CAF50')
]:
    ax.plot(smooth(rewards, 20), linewidth=2, label=label, color=color)

ax.axhline(y=475, color='green', linestyle='--', alpha=0.5, label='Solved')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('REINFORCE vs A2C on CartPole-v1')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. PPO with Stable-Baselines3

**Proximal Policy Optimization (PPO)** (Schulman et al., 2017) is one of the most popular RL algorithms. It uses a clipped surrogate objective to prevent large policy updates:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta) \hat{A}_t,\; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]$$

where $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$.

Stable-Baselines3 provides a well-tuned implementation.

In [ ]:
try:
    from stable_baselines3 import PPO
    from stable_baselines3.common.callbacks import BaseCallback
    from stable_baselines3.common.evaluation import evaluate_policy
    
    SB3_AVAILABLE = True
    print('Stable-Baselines3 loaded successfully.')
except ImportError:
    SB3_AVAILABLE = False
    print('Stable-Baselines3 not installed. Install with: pip install stable-baselines3')
    print('Skipping PPO section -- the rest of the notebook still works.')

In [ ]:
if SB3_AVAILABLE:
    # Custom callback to record episode rewards during training
    class RewardCallback(BaseCallback):
        def __init__(self):
            super().__init__()
            self.episode_rewards = []
            self._current_reward = 0.0
        
        def _on_step(self):
            # Accumulate rewards
            self._current_reward += self.locals['rewards'][0]
            if self.locals['dones'][0]:
                self.episode_rewards.append(self._current_reward)
                self._current_reward = 0.0
            return True
    
    # Train PPO
    env_ppo = gym.make('CartPole-v1')
    
    model_ppo = PPO(
        'MlpPolicy',
        env_ppo,
        learning_rate=3e-4,
        n_steps=256,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        verbose=0,
        seed=42
    )
    
    callback = RewardCallback()
    print('=== Training PPO ===')
    model_ppo.learn(total_timesteps=100000, callback=callback)
    rewards_ppo = callback.episode_rewards
    print(f'PPO trained for {len(rewards_ppo)} episodes.')
    
    # Evaluate
    mean_reward, std_reward = evaluate_policy(model_ppo, env_ppo, n_eval_episodes=20)
    print(f'PPO Evaluation: mean={mean_reward:.1f}, std={std_reward:.1f}')
    env_ppo.close()
else:
    rewards_ppo = None
    print('Skipping PPO training (stable-baselines3 not available).')

## 6. Full Algorithm Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

algorithms = [
    (rewards_reinforce, 'REINFORCE', '#2196F3'),
    (rewards_reinforce_bl, 'REINFORCE + Baseline', '#9C27B0'),
    (rewards_a2c, 'A2C', '#4CAF50'),
]

if rewards_ppo is not None:
    algorithms.append((rewards_ppo, 'PPO (SB3)', '#FF5722'))

for rewards, label, color in algorithms:
    s = smooth(np.array(rewards, dtype=float), 20)
    ax.plot(s, linewidth=2, label=label, color=color)

ax.axhline(y=475, color='green', linestyle='--', alpha=0.4, label='Solved')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (20-episode moving average)')
ax.set_title('Policy Gradient Algorithms Comparison -- CartPole-v1')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Understanding the Policy

Let us visualize how a trained policy assigns action probabilities across different states.

In [ ]:
# Train a fresh A2C for visualization
env_vis = gym.make('CartPole-v1')
state_dim = env_vis.observation_space.shape[0]
n_actions = env_vis.action_space.n

model_vis = ActorCritic(state_dim, n_actions).to(device)
optimizer_vis = optim.Adam(model_vis.parameters(), lr=3e-4)

# Quick training
for ep in range(300):
    state, _ = env_vis.reset(seed=42 + ep)
    log_probs, values, rewards_ep, entropies = [], [], [], []
    done = False
    while not done:
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs, value = model_vis(state_t)
        dist = Categorical(probs)
        action = dist.sample()
        next_state, reward, term, trunc, _ = env_vis.step(action.item())
        done = term or trunc
        log_probs.append(dist.log_prob(action))
        values.append(value)
        rewards_ep.append(reward)
        entropies.append(dist.entropy())
        state = next_state
    
    T = len(rewards_ep)
    returns = torch.zeros(T).to(device)
    G = 0.0
    for t in reversed(range(T)):
        G = rewards_ep[t] + 0.99 * G
        returns[t] = G
    values_t = torch.cat(values).squeeze()
    advantages = returns - values_t.detach()
    if advantages.std() > 0:
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    actor_loss = -(torch.stack(log_probs) * advantages).mean()
    critic_loss = F.mse_loss(values_t, returns)
    loss = actor_loss + 0.5 * critic_loss - 0.01 * torch.stack(entropies).mean()
    optimizer_vis.zero_grad()
    loss.backward()
    optimizer_vis.step()

env_vis.close()

# Visualize: probability of pushing right as function of angle and angular velocity
angles = np.linspace(-0.25, 0.25, 50)
ang_vels = np.linspace(-2, 2, 50)
AA, AV = np.meshgrid(angles, ang_vels)

prob_right = np.zeros_like(AA)
value_map = np.zeros_like(AA)

model_vis.eval()
with torch.no_grad():
    for i in range(len(ang_vels)):
        for j in range(len(angles)):
            state = np.array([0.0, 0.0, AA[i, j], AV[i, j]])
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            probs, val = model_vis(state_t)
            prob_right[i, j] = probs[0, 1].item()
            value_map[i, j] = val.item()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Policy map
ax = axes[0]
im = ax.contourf(np.degrees(AA), AV, prob_right, levels=20, cmap='RdYlBu_r')
plt.colorbar(im, ax=ax, label='P(push right)')
ax.set_xlabel('Pole Angle (degrees)')
ax.set_ylabel('Angular Velocity')
ax.set_title('Policy: P(push right | angle, angular vel)')

# Value map
ax = axes[1]
im = ax.contourf(np.degrees(AA), AV, value_map, levels=20, cmap='viridis')
plt.colorbar(im, ax=ax, label='V(s)')
ax.set_xlabel('Pole Angle (degrees)')
ax.set_ylabel('Angular Velocity')
ax.set_title('Critic: V(s) as function of angle, angular vel')

plt.tight_layout()
plt.show()

## Summary

| Algorithm | Type | Key Idea |
|-----------|------|----------|
| **REINFORCE** | Policy gradient | Monte Carlo returns as gradient signal |
| **REINFORCE + Baseline** | Policy gradient | Subtract baseline to reduce variance |
| **A2C** | Actor-Critic | TD advantage for low-variance updates |
| **PPO** | Actor-Critic | Clipped objective for stable, large updates |

### Key Takeaways

- Policy gradient methods can handle **continuous actions** and **stochastic policies**.
- **Variance reduction** (baselines, critics) is essential for practical performance.
- **PPO** is the current workhorse of practical deep RL (ChatGPT's RLHF, robotics, games).
- The critic's value estimate is crucial -- a bad critic means bad advantages.

**Next:** Day 5 -- Applications, custom environments, multi-agent RL, and topological perspectives